# DoMINO sampling visualization

Runs each case through the **real** DoMINO datapipe (`physicsnemo.datapipes.cae.domino_datapipe.create_domino_dataset`)
-- the exact same object `train.py`/`test.py` iterate over -- and plots which points it
actually sampled from the full surface mesh and full STL geometry, so you can eyeball whether
`model.surface_points_sample`/`model.geom_points_sample` sampling (area-weighted or uniform,
per `model.surface_sampling_algorithm`) is covering each case reasonably, rather than
clustering in one region or missing parts of the geometry.

**Unlike `data_distribution_analysis.ipynb`, this notebook DOES import PhysicsNeMo** -- it
needs the exact same environment `train.py`/`test.py` need (see README "Environment notes" /
"Moving to a real NVIDIA GPU"). On a machine with no NVIDIA GPU, this still runs on CPU (same
as the project's own smoke test), just slower.

**Prerequisites**:
- `compute_statistics.py` must already have been run for the config you point this at (needs
  `data.scaling_factors` to exist) -- same requirement as `train.py`.
- If `nvidia-dali` isn't installed (no real GPU here), the next cell points `sys.path` at
  `devtools/dali_stub` first, same as `tests/run_smoke_test.sh` -- harmless if you *do* have
  real DALI installed, since that would already satisfy the import and the stub is simply
  never reached.

**Cost**: fetching each case runs the *entire* datapipe for that case, including SDF
computation on `model.interp_res`'s grid -- not just the point sampling. That's unavoidable if
we want this to be exactly what training sees rather than a separate reimplementation, but it
means each case can take a while on CPU with a large `interp_res`/mesh. Keep `N_CASES` small
for a quick look; there's no need to visualize every case to spot a sampling problem.

**Phases**: `phase="train"` and `phase="val"` use the identical dataset-construction call
`train.py` uses for each. `phase="test"` isn't wired up here -- `test.py` overrides
`model.surface_points_sample` via `eval.sampling`/`eval.surface_points_sample` for its own
reasons (see conf/config.yaml), so evaluation-time sampling isn't quite the same call; adapt
the config cell below if you need to inspect that specifically.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
# See markdown above: only used if nvidia-dali isn't already importable.
sys.path.insert(0, str(PROJECT_ROOT / "devtools" / "dali_stub"))
# conf/config.yaml's paths (data.scaling_factors, data.input_dir, etc.) are relative,
# resolved the same way train.py/test.py resolve them: relative to the project root
# they're normally launched from, not relative to this notebook's own directory.
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
import zarr
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 -- registers the '3d' projection
from omegaconf import OmegaConf

import cuml_knn_patch  # noqa: F401 -- see that file: GPU-only cuML API workaround, inert on CPU
import geometry_sampling_patch  # noqa: F401 -- see that file: area-weights geom_points_sample; train.py/test.py apply this too, so this notebook shows what training actually samples
from utils import get_keys_to_read, load_scaling_factors
from physicsnemo.datapipes.cae.domino_datapipe import create_domino_dataset
from physicsnemo.distributed import DistributedManager
from physicsnemo.utils.domino.utils import unnormalize

%matplotlib inline

DistributedManager.initialize()

## Configuration

Edit freely -- doesn't have to match what you'd actually train with (e.g. shrink
`model.interp_res` here for a faster look, independent of `conf/config.yaml`).

In [ ]:
cfg = OmegaConf.load(PROJECT_ROOT / "conf" / "config.yaml")

# Which split to sample from: "train" or "val" (see markdown above re: "test").
PHASE = "train"

# How many cases to visualize. Each one runs the full datapipe (SDF included) -- see
# "Cost" above. Cases are taken in dataset order starting from CASE_START.
N_CASES = 3
CASE_START = 0

# CPU-safety: this project's shipped config.yaml defaults to GPU preprocessing/output.
# Auto-disable both if no CUDA device is present, same fallback tests/run_smoke_test.sh
# applies explicitly -- remove this if you're deliberately running on a GPU machine and
# want to keep the config's own settings.
if not torch.cuda.is_available():
    cfg.data.gpu_preprocessing = False
    cfg.data.gpu_output = False

print(f"Phase: {PHASE}")
print(f"surface_points_sample={cfg.model.surface_points_sample}, "
      f"geom_points_sample={cfg.model.geom_points_sample}, "
      f"surface_sampling_algorithm={cfg.model.surface_sampling_algorithm}, "
      f"sampling={cfg.data.sampling}, sample_in_bbox={cfg.data.sample_in_bbox}")

## Build the dataset

Identical construction to `train.py`'s `train_dataloader`/`val_dataloader` (same function,
same arguments) -- this is not a separate reimplementation of sampling, it's the actual
pipeline object `train_epoch`/`val_epoch` iterate over.

In [ ]:
surf_factors = load_scaling_factors(cfg)
keys_to_read, keys_to_read_if_available = get_keys_to_read(cfg, get_ground_truth=True)

dataset = create_domino_dataset(
    cfg,
    phase=PHASE,
    keys_to_read=keys_to_read,
    keys_to_read_if_available=keys_to_read_if_available,
    vol_factors=None,
    surf_factors=surf_factors,
    normalize_coordinates=cfg.data.normalize_coordinates,
    sample_in_bbox=cfg.data.sample_in_bbox,
    sampling=cfg.data.sampling,
)

case_paths = dataset.dataset._filenames
print(f"{len(dataset)} case(s) available in this split")
N_CASES = min(N_CASES, len(dataset) - CASE_START)
print(f"Visualizing {N_CASES} case(s) starting at index {CASE_START}")

## Helper functions

`get_case_data` runs one case through the real pipeline (`dataset[idx]`, exactly what
`train_epoch`/`val_epoch` receive) and also reads that same case's *full*, unsampled arrays
directly from its `.zarr` (the pipeline overwrites `surface_mesh_centers` with the sampled
subset, so the only way to see what was left out is to read the raw file too). Predicted
coordinates are converted back to physical units (`unnormalize`, using the pipeline's own
`surface_min_max`) so the overlay is in the same units as the raw mesh -- geometry points from
`downsample_geometry` are already in physical units, no conversion needed there.

In [ ]:
def get_case_data(dataset, idx):
    """Return dict with both the pipeline's sampled batch and the case's raw full arrays."""
    batch = dataset[idx]
    case_path = dataset.dataset._filenames[idx]

    full_group = zarr.open_group(str(case_path), mode="r")
    full_surface = np.asarray(full_group["surface_mesh_centers"])
    full_surface_field = np.asarray(full_group["surface_fields"]).flatten()
    full_stl = np.asarray(full_group["stl_coordinates"])

    sampled_surface = batch["surface_mesh_centers"]
    if "surface_min_max" in batch:
        sampled_surface = unnormalize(
            sampled_surface, batch["surface_min_max"][:, 1], batch["surface_min_max"][:, 0]
        )
    sampled_surface = sampled_surface[0].cpu().numpy()
    sampled_surface_field = batch["surface_fields"][0, :, 0].cpu().numpy()
    sampled_geom = batch["geometry_coordinates"][0].cpu().numpy()

    return {
        "case_name": Path(case_path).stem,
        "full_surface": full_surface,
        "full_surface_field": full_surface_field,
        "full_stl": full_stl,
        "sampled_surface": sampled_surface,
        "sampled_surface_field": sampled_surface_field,
        "sampled_geom": sampled_geom,
    }


def plot_points_overlay(full_points, sampled_points, sampled_values=None, title="",
                         sampled_label="sampled", sampled_cmap="viridis", sampled_color="tab:red"):
    """4-panel overlay: 3D scatter + XY/XZ/YZ projections. Full mesh in light gray,
    sampled points colored (by sampled_values if given, else a flat color)."""
    fig = plt.figure(figsize=(18, 4.5))
    axes_2d = [(0, 1, "X", "Y"), (0, 2, "X", "Z"), (1, 2, "Y", "Z")]

    ax3d = fig.add_subplot(1, 4, 1, projection="3d")
    ax3d.scatter(*full_points.T, s=1, alpha=0.15, color="gray", label="full mesh")
    if sampled_values is not None:
        sc = ax3d.scatter(*sampled_points.T, s=4, alpha=0.9, c=sampled_values, cmap=sampled_cmap)
    else:
        sc = ax3d.scatter(*sampled_points.T, s=4, alpha=0.9, color=sampled_color)
    ax3d.set_title("3D")
    ax3d.set_xlabel("X"); ax3d.set_ylabel("Y"); ax3d.set_zlabel("Z")

    for panel_idx, (i, j, xi, yj) in enumerate(axes_2d):
        ax = fig.add_subplot(1, 4, panel_idx + 2)
        ax.scatter(full_points[:, i], full_points[:, j], s=1, alpha=0.15, color="gray", label="full mesh")
        if sampled_values is not None:
            ax.scatter(sampled_points[:, i], sampled_points[:, j], s=6, alpha=0.9,
                       c=sampled_values, cmap=sampled_cmap)
        else:
            ax.scatter(sampled_points[:, i], sampled_points[:, j], s=6, alpha=0.9,
                       color=sampled_color, label=sampled_label)
        ax.set_xlabel(xi); ax.set_ylabel(yj)
        ax.set_aspect("equal", adjustable="datalim")
        ax.set_title(f"{xi}{yj} projection")
        if panel_idx == 0:
            ax.legend(loc="upper right", fontsize=8)

    fig.suptitle(
        f"{title}  ({len(sampled_points)} of {len(full_points)} points sampled, "
        f"{100 * len(sampled_points) / len(full_points):.1f}%)",
        y=1.05,
    )
    if sampled_values is not None:
        fig.colorbar(sc, ax=fig.axes, shrink=0.7, pad=0.02, label=sampled_label)
    plt.show()

## Surface point sampling, per case

Sampled points colored by their (physical-units) temperature value -- besides checking
spatial coverage, this also shows whether sampling is missing the hot/cold extremes of the
field rather than just missing a region of the geometry.

In [ ]:
case_data = []
for idx in range(CASE_START, CASE_START + N_CASES):
    data = get_case_data(dataset, idx)
    case_data.append(data)
    plot_points_overlay(
        data["full_surface"],
        data["sampled_surface"],
        sampled_values=data["sampled_surface_field"],
        title=f"[{data['case_name']}] surface_mesh_centers sampling",
        sampled_label="temperature",
    )

## Geometry (STL) point sampling, per case

`model.geom_points_sample` points sampled from `stl_coordinates`, area-weighted by
`geometry_sampling_patch.py` (imported above) -- physicsnemo's own `downsample_geometry`
samples uniformly per-vertex with no area weighting at all, which biases samples toward
whichever regions happen to be meshed with more/smaller triangles, independent of their
actual surface area (e.g. two faces of equal physical area but very different triangle
counts get very different sample counts). No per-point field to color by here, so sampled
points are a flat color -- the check is purely spatial coverage of the geometry.

In [ ]:
for data in case_data:
    plot_points_overlay(
        data["full_stl"],
        data["sampled_geom"],
        sampled_values=None,
        title=f"[{data['case_name']}] geometry_coordinates (STL) sampling",
        sampled_color="tab:orange",
    )

## Per-case sampling summary

In [ ]:
summary = pd.DataFrame([
    {
        "case": d["case_name"],
        "n_surface_full": len(d["full_surface"]),
        "n_surface_sampled": len(d["sampled_surface"]),
        "surface_fraction": len(d["sampled_surface"]) / len(d["full_surface"]),
        "n_stl_full": len(d["full_stl"]),
        "n_geom_sampled": len(d["sampled_geom"]),
        "geom_fraction": len(d["sampled_geom"]) / len(d["full_stl"]),
    }
    for d in case_data
])
summary

## kNN surface-neighbor sanity check

One sampled point from the last case above, plus its `model.num_neighbors_surface` nearest
neighbors (`surface_mesh_neighbors`, what the model actually receives alongside each sampled
point). These should sit visibly close to the chosen point -- if they look scattered/far away,
something's off with the kNN backend (see README "Moving to a real NVIDIA GPU" re: cuML vs.
the brute-force torch fallback) rather than with the point sampling itself.

In [ ]:
batch = dataset[CASE_START + N_CASES - 1]
point_idx = 0  # any index into the sampled points is fine

sampled = batch["surface_mesh_centers"]
neighbors = batch["surface_mesh_neighbors"]
if "surface_min_max" in batch:
    s_max, s_min = batch["surface_min_max"][:, 1], batch["surface_min_max"][:, 0]
    sampled = unnormalize(sampled, s_max, s_min)
    neighbors = unnormalize(neighbors, s_max, s_min)
sampled = sampled[0].cpu().numpy()
neighbors = neighbors[0, point_idx].cpu().numpy()  # (num_neighbors_surface - 1, 3)
chosen_point = sampled[point_idx]

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(projection="3d")
ax.scatter(*sampled.T, s=3, alpha=0.15, color="gray", label="all sampled points")
ax.scatter(*neighbors.T, s=40, color="tab:blue", label="neighbors")
ax.scatter(*chosen_point, s=80, color="tab:red", marker="*", label="chosen point")
ax.set_title(f"{case_data[-1]['case_name']}: point {point_idx} and its surface neighbors")
ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
ax.legend()
plt.show()

## Using this on other configs/splits

Change `PHASE` to `"val"` and re-run from the Configuration cell, or edit `cfg` directly
(e.g. `cfg.data.input_dir = "..."`) before the "Build the dataset" cell -- same override
pattern as running `train.py`/`test.py` with command-line overrides, just set in Python here
instead. `N_CASES`/`CASE_START` control which cases get plotted; increase `N_CASES` for a
broader look once you've confirmed it looks reasonable on a few.